In [ ]:
# ======================================================================
# DUBLIN PPR (processed): Dublin sales only + location columns
# ======================================================================
# Input:  dataSet\processed_dataset\property_price_dataset\ppr_2021_onwards.csv
# Output: just_dublin_dataset\just_dublin_processed_dataset\ppr\ppr_dublin.csv
#
# The district comes from the Eircode area code (D12 = Dublin 12).
# If there is no Eircode, the address is searched for 'Dublin 12'.
# local_authority stays empty: the sales register doesn't say the council.
#
# Needs 00_dublin_location_lookup.ipynb to be run first.
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

ppr_file = base + r'\dataSet\processed_dataset\property_price_dataset\ppr_2021_onwards.csv'
dublin_processed = base + r'\just_dublin_dataset\just_dublin_processed_dataset'
save_folder = dublin_processed + r'\ppr'

os.makedirs(save_folder, exist_ok=True)

In [ ]:
ppr = pd.read_csv(ppr_file, low_memory=False)
print('All of Ireland:', ppr.shape)

In [ ]:
# Keeping only Dublin sales

ppr = ppr[ppr['county'].str.strip() == 'Dublin']
print('Dublin only:', ppr.shape)

In [ ]:
# Eircode area = first 3 characters of the Eircode (e.g. 'D12 X2Y3' -> 'D12')

ppr['eircode_area'] = ppr['eircode'].str.strip().str[:3]

print('Sales with an Eircode:', round(ppr['eircode_area'].notnull().mean() * 100, 1), '%')
ppr['eircode_area'].value_counts().head(30)

In [ ]:
# Eircode area -> district, using the lookup.
# Codes that are not Dublin districts (A94 Blackrock, K67 Swords...)
# don't match and stay empty for now.

lookup = pd.read_csv(dublin_processed + r'\dublin_location_lookup.csv')
# (the 'no district' row has no Eircode, so it is left out of this map;
#  otherwise a missing Eircode would wrongly match it)
with_code = lookup[lookup['eircode_area'].notnull()]
eircode_map = dict(zip(with_code['eircode_area'], with_code['dublin_district']))

ppr['dublin_district'] = ppr['eircode_area'].map(eircode_map)

print('District found from Eircode:', ppr['dublin_district'].notnull().sum())

In [ ]:
# For sales without a district yet, searching the address for 'DUBLIN 12' etc.
# (same pattern as in the planning notebook)

def number_to_district(number):
    if pd.isnull(number):
        return np.nan
    w = ''
    if number.endswith('W'):
        w = 'W'
        number = number[:-1]
    return 'Dublin ' + str(int(number)) + w


address = ppr['address'].fillna('').str.upper()
from_address = address.str.extract(r'DUBLIN\s*(\d+W?)')[0].apply(number_to_district)

still_missing = ppr['dublin_district'].isnull()
print('Filled from address:', (still_missing & from_address.notnull()).sum())

ppr.loc[still_missing, 'dublin_district'] = from_address[still_missing]

In [ ]:
# Checking the Eircode and the address agree where both are present

both = ppr['eircode_area'].map(eircode_map).notnull() & from_address.notnull()
agree = (ppr['eircode_area'].map(eircode_map) == from_address)[both].mean() * 100
print('Eircode and address agree on the district:', round(agree, 1), '%')

In [ ]:
# Anything left over (or not a real district) -> 'Dublin (no district)'

district_list = list(lookup['dublin_district'])
ppr.loc[~ppr['dublin_district'].isin(district_list), 'dublin_district'] = 'Dublin (no district)'

has_district = (ppr['dublin_district'] != 'Dublin (no district)').mean() * 100
print('Sales with a postal district:', round(has_district, 1), '%')
ppr['dublin_district'].value_counts()

In [ ]:
# Adding district_id and the higher levels from the lookup.
# local_authority is left empty (not in the sales register).

ppr['county'] = 'Dublin'
ppr = ppr.merge(lookup[['dublin_district', 'district_id', 'county_id', 'province', 'nation']],
                on='dublin_district', how='left')
ppr['local_authority'] = np.nan

print('Rows without a district id:', ppr['district_id'].isnull().sum())   # should be 0
ppr[['address', 'eircode_area', 'dublin_district', 'district_id', 'county',
     'province', 'nation']].head()

In [ ]:
ppr.to_csv(save_folder + r'\ppr_dublin.csv', index=False)
print('Saved', ppr.shape, 'to', save_folder + r'\ppr_dublin.csv')